In [4]:
%cd dev1/scaling-laws-ecnn/

/home/frischs/dev1/scaling-laws-ecnn


In [22]:
import numpy as np
import os

In [18]:
location = "../../Data/frischs/datasets/cifar10/CIFAR-10-C/"

In [15]:
%ls ../../Data/frischs/datasets/cifar10/CIFAR-10-C/

brightness.npy         gaussian_noise.npy    saturate.npy
contrast.npy           glass_blur.npy        shot_noise.npy
defocus_blur.npy       impulse_noise.npy     snow.npy
elastic_transform.npy  jpeg_compression.npy  spatter.npy
fog.npy                labels.npy            speckle_noise.npy
frost.npy              motion_blur.npy       zoom_blur.npy
gaussian_blur.npy      pixelate.npy


In [21]:
np.load(location + 'labels.npy').shape

(50000,)

In [27]:
files = os.listdir(location)
files = [f.split(".")[0] for f in files if "labels" not in f]
files

['speckle_noise',
 'defocus_blur',
 'brightness',
 'frost',
 'jpeg_compression',
 'glass_blur',
 'gaussian_blur',
 'elastic_transform',
 'shot_noise',
 'spatter',
 'fog',
 'contrast',
 'zoom_blur',
 'saturate',
 'motion_blur',
 'gaussian_noise',
 'pixelate',
 'impulse_noise',
 'snow']

In [33]:
from torch.utils.data.dataset import Dataset
import torch

In [34]:
class CIFAR10_C(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        files = os.listdir(location)
        files = [f.split(".")[0] for f in files if "labels" not in f]
        
        self.test_images_under_dif_perturbations = []
        self.test_labels_under_dif_perturbations = []
        self.meta_data_list = []
        for file in files:
            self.test_images_under_dif_perturbations.append(np.load(location + file + '.npy'))
            self.test_labels_under_dif_perturbations.append(np.load(location + 'labels.npy'))
            self.meta_data_list.append([file] * len(self.test_labels_under_dif_perturbations[-1]))
        
        self.test_images_under_dif_perturbations = np.concatenate(self.test_images_under_dif_perturbations)
        self.test_labels_under_dif_perturbations = np.concatenate(self.test_labels_under_dif_perturbations)
        self.meta_data_list = np.concatenate(self.meta_data_list)

        print("test_images_under_dif_perturbations.shape: ", self.test_images_under_dif_perturbations.shape)
        print("test_labels_under_dif_perturbations.shape: ", self.test_labels_under_dif_perturbations.shape)
        print("meta_data_list.shape: ", self.meta_data_list.shape)

        
    def __len__(self):
        return len(self.test_images_under_dif_perturbations)
    
    def __getitem__(self, index):
        image = self.test_images_under_dif_perturbations[index]
        label = self.test_labels_under_dif_perturbations[index]
        meta_data = self.meta_data_list[index]
        
        image = Image.fromarray(image)
        
        if self.transform is not None:
            image = self.transform(image)
        
        label = torch.tensor(label, dtype=torch.int64)
        
        return image, label, meta_data
    
    def eval(self, predictions, labels, meta_data_list):
        
        meta_data_list = meta_data_list.numpy()
        
        

In [35]:
CIFAR10_C(location)

test_images_under_dif_perturbations.shape:  (950000, 32, 32, 3)
test_labels_under_dif_perturbations.shape:  (950000,)
meta_data_list.shape:  (950000,)


In [ ]:
for p in ['gaussian_noise', 'shot_noise', 'motion_blur', 'zoom_blur',
          'spatter', 'brightness', 'translate', 'rotate', 'tilt', 'scale']: